# Project DAT305 - Motion Regognition from IMU data

By Andreas Nesse & Simen Kolbjørnsen

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.functions import *
from src.classes import *

import keras
import tensorflow

## TODO:

### Administrative
1. Deside on technology:

* Keras & Tensorflow
* Numpy & Pandas


2. Set problem/task boundaries and structure


3. Report


4. Presentation


5. Review

### Implementation
1. Import data and structure data files
2. Load data into notebook
3. Clean dataset
4. Structure the data for the model
5. Implement and structure the model
6. Train the model on the data
7. Feedback loop

### Implementation method:
1 step is ~500 + seconds long on avg
This means, with a $\Delta t$ of 0.0125 s or 12.5 ms between the IMU readings, 1 step is $500 / 12.5 = 40$ points. In each IMU reading we have 6 values, 3 for acceleration and 3 for gyroscope data.

By entering the data raw to the model, this means that the model has to take in 240 input values.

To compress this we can use the magnitude of the acceleration and gyroscope data, meaning that instead of 6 values per point, we get 2. Again leading to 80 input values for the model

Furthermore it could be possible to just use statistical/mathematical properties of the data. Like computing the mean, sd, variance, max, min etc. This again could make the input size less than 10.

We want to make the model as compressed as possible. So converting all values to 8-bit integers instead of the default 64-bit would reduce the size by 1/4. Also using a more traditional approach, like using regression models instead of a NN will also be optimal as long as it manages to give the right results.

We have 2 627 548 data points from 75 different recording sessions from 31 different people. Meaning we have som variety in the data so that the model doesn't get alligned only to one type of person (e.g. tall or short people). So say each input to the model is 40 points, that leaves us with $2 627 548 / 40 = 65 688.7$ inputs of data to be used to train/validate/test. So just over $2^{16}$.



## 1. DATA
### 1.1 Loading the data

The data is stored in multiple csv files in the github repository [`SIMUL-dataset`](https://github.com/simpleLoc/SIMUL-dataset). We scraped the urls for the files and stored the relative urls in [`file_urls.csv`](./file_urls.csv), such that they can be added to the `BASE_URL` provided in the code block below to get the full path. 

We can set the `files_to_read` to the number of files we want to read. Our implementation will maximum read 75 files, as it is the amount of csv files that are stored in the repository.

In [ ]:
files_to_read = 75
max_files = 75
N = max(min(files_to_read, max_files), 1)
try:
    if N != N_loaded:
        raise NameError
    print(f"Data already loaded: {len(data)} files")
except NameError:
    BASE_URL = "https://raw.githubusercontent.com/simpleLoc/SIMUL-dataset/refs/heads/master/data/by-person/"
    
    file_URLs = pd.read_csv("https://raw.githubusercontent.com/andynesse/DAT305/refs/heads/main/file_urls.csv", header=None)[0]
    data = load_data(BASE_URL, file_URLs[:N])
    print(len(data))
N_loaded = N

Select and rename relevant parameters and then do describe on all the dataframes:

In [ ]:
columns = {'Timestamp' : 'ts', ' Activity' : 'activity', ' AccelX_3' : 'aX', ' AccelY_3' : 'aY', ' AccelZ_3' : 'aZ', ' GyroX_3' : 'gX', ' GyroY_3' : 'gY', ' GyroZ_3' : 'gZ'}
for i, df in enumerate(data):
    df = df.rename(columns=columns)
    df = df[list(columns.values())]
    print(df.describe())
    data[i] = df


Check if we have null values (Fortunately we have none for every column in every file):

In [ ]:

print(sum([df.isnull().sum() for df in data]))

Here we plot the magnetude of the acc and gyro for a dataframe:

In [ ]:
data_sample = data[0]

ts = data_sample['ts'].to_numpy()
aX = data_sample['aX'].to_numpy()
aY = data_sample['aY'].to_numpy()
aZ = data_sample['aZ'].to_numpy()
gX = data_sample['gX'].to_numpy()
gY = data_sample['gY'].to_numpy()
gZ = data_sample['gZ'].to_numpy()

plt.plot(ts, magnitude(aX, aY, aZ), label="Acceleration")
plt.plot(ts, magnitude(gX, gY, gZ), label="Gyro")

#plt.xlim(left = 155, right = 160)
plt.legend()
plt.show()



Piechart of how many of the datapoints are walking or still in the df:

In [ ]:
activities = data_sample['activity']
still = activities[activities == 0]
pie = [len(still)/len(activities), 1 - len(still)/len(activities)]
plt.pie(pie, labels=["still", "in motion"],colors=("black","pink"))
plt.show()

A closer look on the magnetude of acc and gyro, so that we can see the repeating signal of walking motion, here we see a 3 second snapshot. Since one step is on average 500ms, this should show around 6 steps:

In [ ]:
data_sample = data[0]

ts = data_sample['ts'].to_numpy()
aX = data_sample['aX'].to_numpy()
aY = data_sample['aY'].to_numpy()
aZ = data_sample['aZ'].to_numpy()
gX = data_sample['gX'].to_numpy()
gY = data_sample['gY'].to_numpy()
gZ = data_sample['gZ'].to_numpy()

plt.plot(ts, magnitude(aX, aY, aZ), label="Acceleration")
plt.plot(ts, magnitude(gX, gY, gZ), label="Gyro")
dt = 3
tx = 106
plt.xlim(left = tx, right = tx+dt)
plt.legend()
plt.show()

From the plot above we can see the spikes in acceleration and gyroscope data when steps are taken.

The amount of datapoints we have, combined from all the files:

In [ ]:
total_datapoints = sum(len(d) for d in data)
print(f"Total datapoints: {total_datapoints}")

To create windows for the data, we made the function [`to_windows`](./src/functions/to_windows.py#L9). It uses one of three window methods:

- [`window_by_parameters`](./src/functions/to_windows.py#L20): Create windows from all values from its columns. Resulting in data being size 40*6 = 240 for input. 
- [`window_by_magnitude`](./src/functions/to_windows.py#L33): Creates windows from the magnitude of the accelerometer data and the gyroscope data for each point. So the resulting size of the data here is 40*2 = 80.
- [`window_by_values`](./src/functions/to_windows.py#L43): Creates windows from statistical properties of the magnitudes. So the resulting size of the data here is 2*7 = 14.

We implemented these methods very generally, so that we easily can swap the number of columns we want to train on.

Our raw data also have 6 different activities:
- standing (0)
- walking (1)
- stairs up (2)
- stairs down(3)
- elevator up (4)
- elevator down (5)

We only want walking and not walking, so we converted activities in [`to_windows.py`](./src/functions/to_windows.py#L7) such that:
- Not walking (0): standing (0), elevator up (4), elevator down (5)
- Walking (1): walking(1), stairs up (2), stairs down (3)

In this way we only have two different output combinations, and thus the model we are going to train becomes much more simpler.

When creating the windoes we want the points in the window to be continous in terms of time, but we also want the points to be of the same label: walking (1) or not walking (0). To achieve this we did the most simple solution, to just drop the windows that are of two different activity labels.

Below we create windows using the three different methods:

In [ ]:
windows_all_df = dfs_to_windows(data, window_by_parameters, 'activity', ['aX', 'aY', 'aZ', 'gX', 'gY', 'gZ'])
windows_all_df.describe()

In [ ]:
windows_magnitudes_df = dfs_to_windows(data, window_by_magnitude, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_magnitudes_df.describe()

In [ ]:
windows_values_df = dfs_to_windows(data, window_by_values, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_values_df.describe()

The amount of datapoints discarded during the windowing process:

In [ ]:
discarded_datapoints = total_datapoints - (len(windows_all_df) * 40)
print(f"Discarded datapoints after windowing: {discarded_datapoints}")
print(f"Fraction of datapoints that were discarded (%): {discarded_datapoints / total_datapoints * 100:.4}%")

## Regression

We want to make some regression models from our 3 different approaches of windowing.

To keep the different window data separated we made a class [`TrainingData`](./src/classes/training_data.py).

In [ ]:
all_points_data = TrainingData(windows_all_df, 'activity')
magnitude_data = TrainingData(windows_magnitudes_df, 'activity')
statistical_data = TrainingData(windows_values_df, 'activity')

#### Make regression model:

In [ ]:
all_points_model = regression_model(all_points_data.n_cols)
magnitude_model = regression_model(magnitude_data.n_cols)
statistical_model = regression_model(statistical_data.n_cols)

Our ``TrainingData` objects have a `fit()` method that takes a model as argument, and then fit the model based on the training data. So using the fit method we can train and test the model at the same time. By default the validation split is 0.3 and running for 100 epochs, however setting the `early_stopping=True` will make the training stop early to prevent overfitting.

In [ ]:
all_points_history = all_points_data.fit(all_points_model, early_stopping=True)

In [ ]:
magnitude_history = magnitude_data.fit(magnitude_model, early_stopping=True)

In [ ]:
statistical_history = statistical_data.fit(statistical_model, early_stopping=True)

We then plot the results in a graph, so that we can see the results side by side:

In [ ]:

pd.DataFrame(all_points_history.history).plot(figsize=(8, 5), grid=True)

pd.DataFrame(magnitude_history.history).plot(figsize=(8, 5), grid=True)

pd.DataFrame(statistical_history.history).plot(figsize=(8, 5), grid=True)
#plt.grid(True)
#plt.gca().set_ylim(0, 1) # set the vertical range to [0-1]
plt.show()

Here we can see the history, showing the loss and val_loss for the epochs in the different models:

In [ ]:
pd.DataFrame(all_points_history.history)

In [ ]:
pd.DataFrame(magnitude_history.history)

In [ ]:
pd.DataFrame(statistical_history.history)

We quantize the model to use 8-bit integers instead of 64-bit floats, so that it should reduce the size of the model by $1/4$. We then save the regression models:

In [ ]:
all_points_model.quantize("int8")
all_points_model.save("all_points_regression_model.keras")

magnitude_model.quantize("int8")
magnitude_model.save("magnitude_regression_model.keras")

statistical_model.quantize("int8")
statistical_model.save("statistical_regression_model.keras")

## Classification model

Now we are going to do the same but with classification instead of regression models:

In [ ]:
all_points_classification_model = classification_model(all_points_data.n_cols)
all_points_classification_history = all_points_data.fit(all_points_classification_model, early_stopping=True)

In [ ]:
magnitude_classification_model = classification_model(magnitude_data.n_cols)
magnitude_classification_history = magnitude_data.fit(magnitude_classification_model, early_stopping=True)

In [ ]:
statistical_classification_model = classification_model(statistical_data.n_cols)
statistical_classification_history = statistical_data.fit(statistical_classification_model, early_stopping=True)

And again see how the results for the different window methods

In [ ]:
pd.DataFrame(all_points_classification_history.history).plot(figsize=(8, 5), grid=True)
pd.DataFrame(magnitude_classification_history.history).plot(figsize=(8, 5), grid=True)
pd.DataFrame(statistical_classification_history.history).plot(figsize=(8, 5), grid=True)
plt.show()

[TODO]

And the history showing loss and validation loss when training the models:

In [ ]:
pd.DataFrame(all_points_classification_history.history)

In [ ]:
pd.DataFrame(magnitude_classification_history.history)

In [ ]:
pd.DataFrame(statistical_classification_history.history)

Again, quantize and save the classification models:

In [ ]:
all_points_classification_model.quantize("int8")
all_points_classification_model.save("all_points_classification_model.keras")

magnitude_classification_model.quantize("int8")
magnitude_classification_model.save("magnitude_classification_model.keras")

statistical_classification_model.quantize("int8")
statistical_classification_model.save("statistical_classification_model.keras")